### Ensemble Learning:
- It combines multiple simpler models to create a stronger, smarter model.
- There are mainly two types of ensemble learning.
  1. bagging - That combines multiple models trained independently.
  2. boosting - That builds models sequentially each correcting the errors of the previous one.

### AdaBoost in Machine Learning:
- AdaBoost is a boosting technique that combines several weak classifiers in sequence of strong one.
- Each new model focus on correcting the mistakes of the previous one until all data is correctly classified (or) a set number of iterations is reached.
### AdaBoost Working:
- AdaBoost assigns equal weights to all training samples initially and iteratively adjusts these weights by focusing more on misclassified data points for the next model.
- It effectively reduces the bias and variance making it useful for classification tasks but it can be sensitive to noisy data and outliers. 

### Implementation of AdaBoost Algorithm:
#### 1. Import Libraries:


In [16]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix,precision_score, recall_score, f1_score, roc_auc_score

#### 2. Defining the AdaBoost Class:
- This class will handle the entire training process and predictions.
- The AdaBoost class is where we define the entire AdaBoost algorithm which consists of:
  - Initializing model parameters like no of estimators, weights and models.
  - Fitting the model to the training data.
  - Making predictions using the trained model.

In [17]:
class AdaBoost:
    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators
        self.alphas = []
        self.models = []

#### 3. Training the AdaBoost Model:
- In the fit()  method we -
- Sample Weighted Initialization:
  - w=np.ones(n_samples)/ n_samples is Initializes all sample weights equally.
- Training the Weak Classifier:
  - A DecisionTreeClassifier with max_depth=1 is trained using the current sample weights.
- Error Calculation:
  - err=np.sum(w*(predictions!=y))/ np.sum(w) computes the weighted error of the classifier.
- Alpha Calculation:
  - alpha = 0.5*np.log((1-err)/(err+1e-10)) calculates the classifier's weight.
- Updated Weights:
  - Misclassified samples weights are increased using w*=np.exp(-alpha* y* predictions) and normalized with w/=np.sum(w).

In [18]:
    def fit(self, X, y):
            n_samples, n_features = X.shape  
            w = np.ones(n_samples) / n_samples 
    
            for _ in range(self.n_estimators):
                model = DecisionTreeClassifier(max_depth=1)  
                model.fit(X, y, sample_weight=w)  
                predictions = model.predict(X)  
    
                err = np.sum(w * (predictions != y)) / np.sum(w)
    
                alpha = 0.5 * np.log((1 - err) / (err + 1e-10))
    
                self.models.append(model) 
                self.alphas.append(alpha)  
    
                w *= np.exp(-alpha * y * predictions)  
                w /= np.sum(w)

#### 4. Defining Predict Method:
- In the predict() method we combine the predictions of all weak classifiers using their respective alpha values to make the final prediction.
- strong_preds = np.zeroes(X.shape[0]):
  - Initializes an array of zeros to store the weighted sum of predictions from all weak classifiers.
- for model, alpha in zip(self.models, self.alphas):
  - loops through each trained model and its corresponding alpha value.
- strong_preds += alpha * predictions:
  - adds the weighted of predicted of each weak model to strong_preds.
- np.sign(strong_preds):
  - takes the sign of the sum to classify samples as 1[positive class] or -1[negative class].

In [19]:
    def predict(self, X):
            strong_preds = np.zeros(X.shape[0])  
    
            for model, alpha in zip(self.models, self.alphas):
                predictions = model.predict(X)  
                strong_preds += alpha * predictions  
    
            return np.sign(strong_preds).astype(int)

#### 5. Example:
- We are generating a synthetic dataset with 1000 samples and 20 features.
- Then, we split the data into training and testing sets.
- We initialize and train an AdaBoost classifier with 50 estimators.
- After training, we predict on the test set and evaluate the model.

In [20]:
if __name__ == "__main__":

    X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    adaboost = AdaBoost(n_estimators=50)
    adaboost.fit(X_train, y_train)

    predictions = adaboost.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    try:
        roc_auc = roc_auc_score(y_test, predictions)
    except ValueError:
        roc_auc = 'Undefined (requires probability scores)'

    print(f"Accuracy: {accuracy * 100}%")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")
    print(f"ROC-AUC: {roc_auc}")

AttributeError: 'AdaBoost' object has no attribute 'fit'